In [8]:
import html as html_lib
import os
import folium
from folium.plugins import MarkerCluster
import numpy as np
import pandas as pd


#--------#

def convert_dms_series(deg, mn, sc, direction):
  """Convertit des colonnes de coordonnées DMS (Degrés, Minutes, Secondes)

  en degrés décimaux de manière vectorisée avec pandas/numpy.
  """
  d = pd.to_numeric(deg, errors="coerce").fillna(0.0)
  m = pd.to_numeric(mn, errors="coerce").fillna(0.0)
  s = pd.to_numeric(sc, errors="coerce").fillna(0.0)
  dec = d + (m / 60.0) + (s / 3600.0)

  # Applique un signe négatif pour le Sud (S) et l'Ouest (W)
  sign = np.where(
      direction.astype(str).str.strip().str.upper().isin(["S", "W"]), -1.0, 1.0
  )
  return dec * sign


def parse_alt(val):
  """Convertit la hauteur AER_NB_ALT_BAS en nombre flottant (float).

  Gère les virgules décimales, les espaces et les valeurs manquantes.
  """
  if pd.isna(val):
    return None
  s = str(val).strip().replace(",", ".")
  try:
    return float(s)
  except ValueError:
    return None


def get_color_from_ratio(ratio):
  """Calcule une couleur hexadécimale sur un dégradé bleu (#3186cc) -> orange (#d95f02)

  selon un ratio compris entre 0.0 (date la plus ancienne) et 1.0 (date la plus récente).
  """
  ratio = max(0.0, min(1.0, ratio))
  r = int(49 + (217 - 49) * ratio)
  g = int(134 + (95 - 134) * ratio)
  b = int(204 + (2 - 204) * ratio)
  return f"#{r:02x}{g:02x}{b:02x}"


def add_legend(m, titre_carte, min_date_str, max_date_str, total_count, max_alt):
  """Ajoute un encadré de légende fixe en bas à gauche de la carte Folium."""
  alt_txt = f"{max_alt:g} m" if max_alt is not None else "—"
  html = f"""
    <div style="
        position: fixed;
        bottom: 24px;
        left: 24px;
        z-index: 9999;
        background-color: white;
        padding: 10px 12px;
        border: 1px solid #ccc;
        border-radius: 6px;
        font-family: Arial, sans-serif;
        font-size: 12px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    ">
        <div style="font-weight: bold; margin-bottom: 3px;">{titre_carte} — 2026 ({total_count})</div>
        <div style="font-size: 11px; color: #dc2626; font-weight: bold; margin-bottom: 6px;">
            <span style="display:inline-block;width:10px;height:10px;background:#dc2626;border-radius:50%;margin-right:4px;"></span>
            Plus haute : {alt_txt}
        </div>
        <div style="font-size: 11px; color: #555; margin-bottom: 5px;">Date de mise en service</div>
        <div style="
            width: 190px;
            height: 12px;
            background: linear-gradient(to right, #3186cc, #d95f02);
            border-radius: 3px;
            margin-bottom: 4px;
        "></div>
        <div style="display: flex; justify-content: space-between; font-size: 11px; color: #444;">
            <span>{min_date_str}</span>
            <span>{max_date_str}</span>
        </div>
    </div>
    """
  m.get_root().html.add_child(folium.Element(html))


def get_lieu(row):
  """Reconstitue l'adresse, le lieu-dit et le code postal sans doublons ni 'nan'."""
  morceaux = []
  for col in ["ADR_LB_ADD1", "ADR_LB_ADD2", "ADR_LB_ADD3", "ADR_LB_LIEU"]:
    val = str(row.get(col, "") or "").strip()
    if val and val.lower() != "nan" and val not in morceaux:
      morceaux.append(val)

  cp = str(row.get("ADR_NM_CP", "") or "").strip()
  if cp and cp.lower() != "nan":
    morceaux.append(cp)

  return " — ".join(morceaux) if morceaux else "Lieu non renseigné"


#--------#


def build_island_map(df_ile, cfg, idx_plus_haute, max_alt):
  """Crée une carte Folium verrouillée sur l'emprise d'une île.

  - Place les antennes standards dans un MarkerCluster coloré selon la date de mise en service.
  - Place l'antenne la plus haute (AER_NB_ALT_BAS) avec un marqueur rouge dédié.
  """
  lat_min, lon_min, lat_max, lon_max = cfg["bbox"]
  center = [(lat_min + lat_max) / 2.0, (lon_min + lon_max) / 2.0]

  # Verrouille le zoom et les bornes géographiques sur l'île
  m = folium.Map(
      location=center,
      zoom_start=cfg["min_zoom"],
      min_zoom=cfg["min_zoom"],
      max_zoom=cfg["max_zoom"],
      max_bounds=True,
      min_lat=lat_min,
      max_lat=lat_max,
      min_lon=lon_min,
      max_lon=lon_max,
      tiles=None,
  )
  m.fit_bounds([[lat_min, lon_min], [lat_max, lon_max]])

  # Ajout des fonds de carte Esri sélectionnables via le contrôle de couches
  for name, url in [
      (
          "Esri Street Map",
          "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
      ),
      (
          "Esri Topo Map",
          "https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
      ),
      (
          "Esri Satellite",
          "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
      ),
  ]:
    folium.TileLayer(
        tiles=url,
        attr="Esri",
        name=name,
        min_zoom=cfg["min_zoom"],
        max_zoom=cfg["max_zoom"],
    ).add_to(m)

  # Calcul de la plage temporelle 2026 sur l'île pour le dégradé de couleur
  dates_valides = df_ile["Mise en service"].dropna()
  if not dates_valides.empty:
    d_min = dates_valides.min()
    d_max = dates_valides.max()
    tmin = d_min.timestamp()
    tmax = d_max.timestamp()
    ecart = (tmax - tmin) if (tmax - tmin) > 0 else 1.0
    add_legend(
        m,
        cfg["nom"],
        d_min.strftime("%Y-%m-%d"),
        d_max.strftime("%Y-%m-%d"),
        len(df_ile),
        max_alt,
    )
  else:
    tmin = 0.0
    ecart = 1.0
    add_legend(m, cfg["nom"], "—", "—", 0, max_alt)

  # Groupe principal regroupé (cluster) + groupe séparé pour l'antenne la plus haute
  cluster = MarkerCluster(name=f"{cfg['nom']} — 2026").add_to(m)
  groupe_plus_haute = folium.FeatureGroup(
      name=f"Plus haute ({cfg['nom']})", show=True
  ).add_to(m)

  for idx, r in df_ile.iterrows():
    est_plus_haute = idx == idx_plus_haute
    date_val = r.get("Mise en service")

    # Détermine la couleur selon la date de mise en service
    if pd.notna(date_val):
      ratio = (date_val.timestamp() - tmin) / ecart
      couleur = get_color_from_ratio(ratio)
      date_str = date_val.strftime("%Y-%m-%d")
    else:
      couleur, date_str = "#666666", "Non renseignée"

    # Prépare les informations affichées dans l'infobulle et le popup
    alt_val = r.get("alt_bas")
    alt_str = f"{alt_val:g} m" if pd.notna(alt_val) else "—"
    lieu = get_lieu(r)
    territoire = cfg["nom"]

    etiquette_info = (
        f"Lieu : {lieu} | Territoire : {territoire} | Hauteur : {alt_str} | "
        f"Mise en service : {date_str}"
    )

    popup_html = f"""
        <div style="font-family: Arial, sans-serif; font-size: 12px; min-width: 230px;">
            <strong>Lieu :</strong> {lieu}<br>
            <strong>Territoire :</strong> {territoire}<br>
            <strong>Hauteur :</strong> <b>{alt_str}</b><br>
            <strong>Mise en service :</strong> {date_str}
        </div>
        """

    if est_plus_haute:
      # Marqueur rouge distinctif pour l'antenne la plus haute de l'île
      folium.Marker(
          location=[r["latitude"], r["longitude"]],
          icon=folium.Icon(color="red", icon="arrow-up", prefix="fa"),
          tooltip=etiquette_info,
          popup=folium.Popup(popup_html, max_width=320),
      ).add_to(groupe_plus_haute)
    else:
      # Cercle coloré pour les autres antennes
      folium.CircleMarker(
          location=[r["latitude"], r["longitude"]],
          radius=6,
          color=couleur,
          weight=1.5,
          fill=True,
          fill_color=couleur,
          fill_opacity=0.85,
          tooltip=etiquette_info,
          popup=folium.Popup(popup_html, max_width=320),
      ).add_to(cluster)

  folium.LayerControl().add_to(m)
  return m.get_root().render()
#--------#

DATA_DIR = r"C:\Users\bouyejay\Desktop\data\data"
SUPPORT_FILE = os.path.join(DATA_DIR, "sup_support.txt")
STATION_FILE = os.path.join(DATA_DIR, "sup_station.txt")
ANTENNE_FILE = os.path.join(DATA_DIR, "sup_antenne.txt")
OUTPUT_GALLERY_HTML = os.path.join(DATA_DIR, "cartes_iles_2026_galerie.html")

# Liste des îles : clé HTML, préfixe code postal, nom, [lat_min, lon_min, lat_max, lon_max], zoom_min, zoom_max
ILES_2026 = [
    {
        "cle": "guadeloupe",
        "prefix": "971",
        "nom": "Guadeloupe",
        "bbox": [15.80, -61.85, 16.55, -61.00],
        "min_zoom": 10,
        "max_zoom": 17,
    },
    {
        "cle": "martinique",
        "prefix": "972",
        "nom": "Martinique",
        "bbox": [14.35, -61.25, 14.90, -60.75],
        "min_zoom": 10,
        "max_zoom": 17,
    },
    {
        "cle": "reunion",
        "prefix": "974",
        "nom": "La Réunion",
        "bbox": [-21.40, 55.20, -20.85, 55.85],
        "min_zoom": 10,
        "max_zoom": 17,
    },
    {
        "cle": "mayotte",
        "prefix": "976",
        "nom": "Mayotte",
        "bbox": [-13.05, 45.00, -12.60, 45.32],
        "min_zoom": 11,
        "max_zoom": 17,
    },
    {
        "cle": "nouvelle_caledonie",
        "prefix": "988",
        "nom": "Nouvelle-Calédonie",
        "bbox": [-22.75, 163.50, -19.50, 168.10],
        "min_zoom": 7,
        "max_zoom": 17,
    },
]

#--------#